### 해양수산부 국립해양조사원_국가해양관측망 실측 파랑
*from 해양수산부 국립해양조사원*
>국가해양관측망(해양관측부이, 해양과학기지 등)에서 관측하는 파랑(파고, 파주기, 파향) 데이터를 제공 기능

- **제공기관:** 해양수산부 국립해양조사원
- **API 유형:** REST	데이터포맷	JSON+XML
- **End Point:** *https://apis.data.go.kr/1192136/noonWave*
- **Service:**  */GetNoonWaveApiService*  [ 파랑(파고, 파주기, 파향) 데이터 ]

In [89]:
%useLatestDescriptors
%use dataframe

In [111]:
val serviceKeyFilePath = "/Users/unchil/.EnvDataCollector/envDataCollector.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [117]:
@file:DependsOn("org.xerial:sqlite-jdbc:3.49.1.0")
import java.sql.DriverManager
Class.forName("org.sqlite.JDBC")

class org.sqlite.JDBC

In [90]:
val filePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/KotlinNotebooks/data/wave_Observatory_list.csv"
val df_observatory = DataFrame.readCsv(filePath, ',')

In [105]:
df_observatory.columnNames()

[code, name]

In [116]:
val tableName_Observatory = "WaveInfoObservatory"
val sql_Observatory = """INSERT INTO $tableName_Observatory (code, name) VALUES (?,?)""".trimIndent()

DriverManager.getConnection(configData.SQLITE_DB.jdbcURL).use { conn ->
    conn.autoCommit = false // 트랜잭션 자동 커밋 비활성화
    conn.prepareStatement(sql_Observatory)?.use { pstmt ->
        df_observatory.forEach { row ->
            try {
                pstmt.setString(1, row["code"].toString())
                pstmt.setString(2, row["name"].toString())
                pstmt.addBatch()
            } catch (e: Exception) {
                println("Batch 생성 에러: ${e.localizedMessage}")
            }
        }
        pstmt.executeBatch()
    }
    conn.commit() // 일괄 커밋
}

In [118]:
val endPoint = configData.WAVE_INFO.endPoint
val serviceKey = configData.WAVE_INFO.apikey
val subPath = configData.WAVE_INFO.subPath
val pageNo = "1"
val numOfRows = "300"
val type = "json"
val obsCode = "TW_0069"
val reqDate = "20261006"
val min = "5"


In [93]:
val url = "${endPoint}/${subPath}?serviceKey=${serviceKey}&pageNo=$pageNo&numOfRows=$numOfRows&type=$type&regDate=${reqDate}&min=$min&obsCode=$obsCode"

In [104]:
val df_waveInfo = DataFrame.readJson(url)

In [95]:
val response = "resultCode:${df_waveInfo.header.resultCode[0]}, totalCount:${df_waveInfo.body.totalCount[0]}"

In [96]:
val resultSet = df_waveInfo.body.items.item.concat()

In [100]:
resultSet.columnNames()

[obsvtrNm, lot, lat, obsrvnDt, wvhgt, wvpd, wvdrct, maxWvhgt, maxWvpd]

In [101]:
val tableName = "WaveInfo"
val sql = """INSERT INTO $tableName (obsvtrNm, lot, lat, obsrvnDt, wvhgt, wvpd, wvdrct, maxWvhgt, maxWvpd) VALUES (?,?,?,?,?,?,?,?,?)""".trimIndent()


In [ ]:
DriverManager.getConnection(configData.SQLITE_DB.jdbcURL).use { conn ->
    conn.autoCommit = false // 트랜잭션 자동 커밋 비활성화
    conn.prepareStatement(sql)?.use { pstmt ->
        resultSet.forEach { row ->
            try {
                pstmt.setString(1, row["obsvtrNm"]?.toString())
                pstmt.setString(2, row["lot"]?.toString())
                pstmt.setString(3, row["lat"]?.toString())
                pstmt.setString(4, row["obsrvnDt"]?.toString())
                pstmt.setString(5, row["wvhgt"]?.toString())
                pstmt.setString(6, row["wvpd"]?.toString())
                pstmt.setString(7, row["wvdrct"]?.toString())
                pstmt.setString(8, row["maxWvhgt"]?.toString())
                pstmt.setString(9, row["maxWvpd"]?.toString())
                pstmt.addBatch()
            } catch (e: Exception) {
                println("Batch 생성 에러: ${e.localizedMessage}")
            }
        }
        pstmt.executeBatch()
    }
    conn.commit() // 일괄 커밋
}